In [8]:
!python --version

Python 3.12.9


## Library

In [15]:
import os
import pandas as pd
import pythainlp 

import gtts 

In [10]:
try:
    import torch
    print("มี torch อยู่ไม่มืดแร้ว" "version:", torch.__version__ if hasattr(torch, '__version__') else "ไม่พบเวอร์ชั่น" )
except ImportError:
    print("อ้ายบ่มีtorch")
    

มี torch อยู่ไม่มืดแร้วversion: 2.9.1


## Data

In [14]:
current_path = os.getcwd()
print("ตอนนี้อยู่ใน:" ,{current_path})

dialogue2 = "Dialogue2.csv"

def dialogue(path):
    df = pd.read_csv(path)
    return df

df = dialogue(dialogue2)
print("โหลดข้อมูลสำเร็จ! พร้อมส่งต่อให้แต่ละโมเดลแล้วค่ะ")
df.head(3)


ตอนนี้อยู่ใน: {'/Users/bam/AIDA-chatbot/backend/voice/dialogue2'}
โหลดข้อมูลสำเร็จ! พร้อมส่งต่อให้แต่ละโมเดลแล้วค่ะ


,ID,Category,Length_Type,Scenario,Keyword,Original Script,TTS Script,Emotion
0,G01,Greeting,Long,ทักทายครั้งแรกแบบ1,"สวัสดี, Hello, หวัดดี, สวัสดีค่ะ, สวัสดีครับ, ...",สวัสดีค่า ไอด้านะคะ ยินดีที่ได้รู้จักไอด้าเป็น...,"สวัสดีค่า, ไอด้านะคะ. ยินดีที่ได้รู้จัก, ไอด้า...",Happy
1,G02,Greeting,Long,ทักทายครั้งแรกแบบ2,"สวัสดี, Hello, หวัดดี, สวัสดีค่ะ, สวัสดีครับ, ...",สวัสดีนะ ยินดีที่ได้รู้จักค่ะ ก่อนอื่นขอแนะนำต...,"สวัสดีนะ, ยินดีที่ได้รู้จักค่ะ. ก่อนอื่นขอแนะน...",Talking
2,G03,Greeting,Short,ทักทายแบบที่3,"สวัสดี, Hello, หวัดดี, สวัสดีค่ะ, สวัสดีครับ, ...",Hello สวัสดีนะคะ ไอด้าเอง ยินดีช่วยเหลือค่ะ สง...,"เฮลโล, สวัสดีนะคะ. ไอด้าเอง, ยินดีช่วยเหลือค่ะ...",Happy


## Text Preprocessing using Pythainlp
1. Count each ID to defining a length type for each sentenses ()
2. To improve context before use voice model (Phoneme/Token Preparation)

explaination: ช่วยลดความลำเอียง (Bias) ของผู้วิจัย และทำให้งานวิจัยนี้สามารถทำซ้ำ (Reproduce)
โมเดลนี้เข้ามาช่วยแยกคำเผื่อที่จะจัดเรียงให้โมเดลเสียงอ่านเข้าใจประโยคหรือคำประสมได้มากขึ้น
1. Short: ทักทายทั่วไปพูดคุยทั่วไปที่ไม่ได้ลงรายละเอียดมาก Low Latency
2. Medium: แทนการตอบคำถามทั่วไปในชีวิตประจำวัน 
3. Long: แทนการอธิบายข้อมูลทางเทคนิคหรือวิชาการที่มีความซับซ้อน

![Response Times](JakobNielsen.png)

In [ ]:
import pandas as pd
import re
from pythainlp import word_tokenize

def get_word_count(text):
    if pd.isna(text): return 0
    # 1. Clean: เก็บไว้แค่ ตัวเลข, อังกฤษ, ไทย และช่องว่าง
    clean_text = re.sub(r'[^0-9a-zA-Z\u0e00-\u0e7f\s]', '', str(text))
    # 2. Tokenize: ตัดคำ
    tokens = word_tokenize(clean_text, engine="newmm")
    return len(tokens)

def classify_length(count):
    if count <= 15: return "Short"
    elif count <= 40: return "Medium"
    else: return "Long"

# --- ขั้นตอนการใช้งาน ---
df['word_count'] = df['Original Script'].apply(get_word_count)
df['Length_Type'] = df['word_count'].apply(classify_length)
print(df[['Original Script', 'word_count', 'Length_Type']])

                                      Original Script  word_count Length_Type
0   สวัสดีค่า ไอด้านะคะ ยินดีที่ได้รู้จักไอด้าเป็น...          31      Medium
1   สวัสดีนะ ยินดีที่ได้รู้จักค่ะ ก่อนอื่นขอแนะนำต...          43        Long
2   Hello สวัสดีนะคะ ไอด้าเอง ยินดีช่วยเหลือค่ะ สง...          19      Medium
3    ขอโทษนะค้า เรื่องนี้พี่ไอด้ายังไม่มีข้อมูลเลยค่ะ          14       Short
4   เรื่องนี้ไอด้าไม่สามารถให้ข้อมูลได้จริงๆค่ะ ขอ...          15       Short
5   ขอโทษน่า มันอยู่นอกเหนือขอบเขตความรับผิดชอบของ...          23      Medium
6   สาขาเราคือ AI และ Data Science  ย่อมาจาก Artif...          26      Medium
7   สาขาของเราเป็นหลักสูตรแรกและหลักสูตรเดียวในไทย...          33      Medium
8   สาขาของเราตั้งอยู่ ณ ตึกวิศวกรรมศาสตร์หรือเรีย...          27      Medium
9             เราใช้ภาษา python เป็นหลักค่ะในการเรียน          10       Short
10  ไอด้าย่อมาจาก Artificial Intelligent and Data ...          14       Short
